In [ ]:
# Ensure Internet toggle on the right side panel is switched ON!
!nvidia-smi

In [ ]:
%%bash
# 1. Clean up any corrupted directory versions cleanly
rm -rf /kaggle/working/spec-fastgs

# 2. Recursively clone using the correct branch name structure
git clone --recursive -b main https://github.com/0Nguyen0Cong0Tuan0/thesis-all.git /kaggle/working/spec-fastgs

# 3. Verify the layout structure
echo "📂 Verifying submodules directory contents:"
ls -la /kaggle/working/spec-fastgs/spec-fastgs/submodules/

In [ ]:
%%bash
# Mirror datasets from the read-only input mount into the WRITABLE working dir
# USING SYMLINKS, not full copies.
#
# Why: some code paths need to write into the scene directory itself --
# scene/dataset_readers.py caches sparse/0/points3D.ply next to the COLMAP
# .bin/.txt files (mip360 scenes), writes points3d.ply for synthetic/Blender
# scenes, and extract_reflection_prior.py writes a new reflection_prior/
# subfolder -- and /kaggle/input is read-only, so SOME writable location is
# required. But a full `cp -r` duplicates every image + COLMAP model byte for
# byte into /kaggle/working's ~19.5GB quota, which is what caused the earlier
# OOM (9 mip360 scenes at images_4 does not fit twice).
#
# Fix: `cp -rs` recreates the directory TREE as real (writable) directories,
# but makes each individual FILE a symlink back to the read-only original in
# /kaggle/input, instead of copying its bytes. Reading a file through a
# symlink is completely transparent (no code changes needed anywhere else),
# and every write this pipeline does (points3D.ply, points3d.ply,
# reflection_prior/*.png, backups/) is a BRAND NEW file/folder that doesn't
# already exist in the source, so it lands as a real file in the (real)
# destination directory without needing to overwrite a symlink. Net effect:
# ~0 extra disk for the dataset mirror instead of ~1x its full size.
WORK=/kaggle/working/spec-fastgs/spec-fastgs/datasets
mkdir -p "$WORK/mipnerf360"

# Mip-NeRF 360 -- CONFIRMED layout (2026-07-13, via screenshot of the actual Kaggle
# Input panel): the 9 scenes are split across two parent folders under one dataset
# mount, not one flat mipnerf360/ folder:
#   /kaggle/input/datasets/thnhdg/testing/360_v2/{bicycle,bonsai,counter,garden,
#                                                  kitchen,room,stump}
#   /kaggle/input/datasets/thnhdg/testing/360_extra_scenes/{flowers,treehill}
# Both get flattened into ./datasets/mipnerf360/<scene> so run_mip360.sh (which
# expects DATA_ROOT=./datasets/mipnerf360 with one subfolder per scene) doesn't
# need to know about the 360_v2/360_extra_scenes split at all.
MIP360_SRC_ROOT=/kaggle/input/datasets/thnhdg/testing

# Recreate the directory TREE for real (writable dirs), but make every
# individual FILE a symlink back to its read-only original -- using explicit
# absolute-path ln -s, not `cp -rs`. GNU cp's --symbolic-link mode computes
# relative symlink targets from the current working directory and fails with
# "can make relative symbolic links only in current directory" on multi-level
# recursive copies where cwd != the destination -- a real GNU coreutils
# limitation (reproduced here), not a Windows/MSYS-only quirk, so it would
# fail the same way on Kaggle's Linux runners. Absolute-target symlinks via
# find+ln sidestep this entirely.
mirror_symlink() {
    local src="$1"
    local dst="$2"
    local src_abs
    src_abs=$(cd "$src" && pwd)
    mkdir -p "$dst"
    find "$src_abs" -type d -print0 | while IFS= read -r -d "" d; do
        mkdir -p "$dst${d#$src_abs}"
    done
    find "$src_abs" -type f -print0 | while IFS= read -r -d "" f; do
        ln -sf "$f" "$dst${f#$src_abs}"
    done
}

copy_scene() {
    local parent="$1"
    local scene="$2"
    local src="${MIP360_SRC_ROOT}/${parent}/${scene}"
    local dst="$WORK/mipnerf360/${scene}"
    if [ -d "$src" ]; then
        echo "📥 mirroring (symlink) $src -> $dst"
        mirror_symlink "$src" "$dst"
    else
        echo "⚠️  ${scene} NOT found at $src"
    fi
}

if [ -d "$MIP360_SRC_ROOT" ]; then
    for scene in bicycle bonsai counter garden kitchen room stump; do
        copy_scene "360_v2" "$scene"
    done
    for scene in flowers treehill; do
        copy_scene "360_extra_scenes" "$scene"
    done
else
    echo "⚠️  ${MIP360_SRC_ROOT} not found -- falling back to generic auto-discovery"
    echo "   (dataset attachment/path may have changed since 2026-07-13; dumping"
    echo "   /kaggle/input up to 6 levels so the path above can be fixed by hand):"
    find /kaggle/input -maxdepth 6 | sort

    # Old generic fallback: locate a flat mipnerf360/ folder by name, or anchor on
    # counter/images if the parent folder is named/nested differently.
    FALLBACK_SRC=$(find /kaggle/input -maxdepth 10 -iname "mipnerf360" -type d 2>/dev/null | head -1)
    if [ -z "$FALLBACK_SRC" ]; then
        COUNTER_IMAGES=$(find /kaggle/input -maxdepth 12 -type d -ipath "*counter/images" 2>/dev/null | head -1)
        if [ -n "$COUNTER_IMAGES" ]; then
            FALLBACK_SRC=$(dirname "$(dirname "$COUNTER_IMAGES")")
        fi
    fi
    if [ -n "$FALLBACK_SRC" ]; then
        echo "📥 fallback: copying $FALLBACK_SRC -> $WORK/mipnerf360"
        for entry in "$FALLBACK_SRC"/*/; do
            scene_name=$(basename "$entry")
            mirror_symlink "$entry" "$WORK/mipnerf360/$scene_name"
        done
    fi
fi

echo "📂 mipnerf360 now in working:"; ls "$WORK/mipnerf360"

# Synthetic suites -- auto-locate under /kaggle/input regardless of the dataset slug
# (location not covered by the confirmed screenshot above, so keep the generic search).
for d in Anisotropic-Synthetic-Dataset Synthetic_NSVF; do
    SRC=$(find /kaggle/input -maxdepth 6 -iname "$d" -type d 2>/dev/null | head -1)
    if [ -n "$SRC" ]; then
        echo "📥 copying $SRC -> $WORK/"
        mirror_symlink "$SRC" "$WORK/$(basename "$SRC")"
    else
        echo "⚠️  $d NOT found under /kaggle/input"
    fi
done
echo "📂 datasets now in working:"; ls "$WORK"


In [ ]:
%%bash
# Clean existing conda toolchain directories safely
rm -rf /opt/conda

# Reinstall isolated Miniconda (Python 3.10)
wget -q https://repo.anaconda.com/miniconda/Miniconda3-py310_23.11.0-1-Linux-x86_64.sh
bash Miniconda3-py310_23.11.0-1-Linux-x86_64.sh -b -p /opt/conda

# Activate custom installation
source /opt/conda/bin/activate

# Install compiler dependencies and CUDA Toolkit 11.7 matching constraints
/opt/conda/bin/conda install -y -c conda-forge cudatoolkit-dev=11.7 gcc_linux-64=11 gxx_linux-64=11

# Export local paths
export CUDA_HOME=/opt/conda
export PATH=$CUDA_HOME/bin:$PATH

echo "----- NVCC -----"
nvcc --version
echo "----- PTXAS -----"
ptxas --version

In [ ]:
%%bash
/opt/conda/bin/pip install torch==1.13.1+cu117 torchvision==0.14.1+cu117 \
  --index-url https://download.pytorch.org/whl/cu117

In [ ]:
%%bash
/opt/conda/bin/python - << 'EOF'
import torch
print("Torch version:", torch.__version__)
print("CUDA back-end:", torch.version.cuda)
print("GPU Available:", torch.cuda.is_available())
EOF

In [ ]:
%%bash
export CUDA_HOME=/opt/conda
export PATH=$CUDA_HOME/bin:$PATH
export LD_LIBRARY_PATH=$CUDA_HOME/lib:$LD_LIBRARY_PATH
export CC=gcc-11
export CXX=g++-11
export CUDAHOSTCXX=g++-11

BASE_DIR="/kaggle/working/spec-fastgs/spec-fastgs/submodules"

# 1. diff-gaussian-rasterization
cd "$BASE_DIR/diff-gaussian-rasterization_fastgs"
rm -rf build dist *.egg-info
/opt/conda/bin/pip install -v .

# 2. simple-knn
cd "../simple-knn"
rm -rf build dist *.egg-info
/opt/conda/bin/pip install -v .

# 3. fused-ssim
cd "../fused-ssim"
rm -rf build dist *.egg-info
/opt/conda/bin/pip install -v .

In [ ]:
%%bash
/opt/conda/bin/python - << 'EOF'
import diff_gaussian_rasterization_fastgs
import simple_knn
import fused_ssim
print("✅ FastGS CUDA extensions compiled and loaded successfully!")
EOF

In [ ]:
%%bash
export CUDA_HOME=/opt/conda
export PATH=$CUDA_HOME/bin:$PATH
export LD_LIBRARY_PATH=$CUDA_HOME/lib:$LD_LIBRARY_PATH
export CC=gcc-11
export CXX=g++-11
export CUDAHOSTCXX=g++-11

BASE_DIR="/kaggle/working/spec-fastgs/spec-fastgs/submodules"

cd "$BASE_DIR/diff-gaussian-rasterization_fastgs"
/opt/conda/bin/python setup.py bdist_wheel

cd "../simple-knn"
/opt/conda/bin/python setup.py bdist_wheel

cd "../fused-ssim"
/opt/conda/bin/python setup.py bdist_wheel

In [ ]:
%%bash
SRC="/kaggle/working/spec-fastgs/spec-fastgs/submodules"
DEST="/kaggle/working/fastgs_wheels_py310"

mkdir -p "$DEST"
find "$SRC" -name "*.whl" -exec cp {} "$DEST" \;

echo "✨ Wheels safely compiled and extracted to: $DEST"
ls -la "$DEST"

In [ ]:
%%bash
/opt/conda/bin/pip uninstall -y numpy
/opt/conda/bin/pip install "numpy<2" plyfile websockets tqdm imageio

In [ ]:
%%bash
/opt/conda/bin/python -c "import fused_ssim; import diff_gaussian_rasterization_fastgs; import plyfile; print('🎉 All systems functional and ready for execution!')"

In [ ]:
# ============================================================
# VERIFY MIP-NERF 360 DATASET LAYOUT (images_4, all 9 scenes)
# ============================================================
# Sanity gate before the multi-hour run_mip360.sh sweep below. Does not
# assume an exact /kaggle/input layout -- just checks what actually landed
# in the working copy after the dataset-copy cell above, so a bad Kaggle
# dataset attachment fails loudly here instead of silently mid-sweep.

import os
import shutil

REPO_ROOT = "/kaggle/working/spec-fastgs/spec-fastgs"
DATASETS_DIR = os.path.join(REPO_ROOT, "datasets")

# Same double-nesting guard the run cell below uses -- some dataset-copy
# strategies leave a stray extra datasets/datasets/ level.
nested = os.path.join(DATASETS_DIR, "datasets")
if os.path.isdir(nested):
    print("Re-aligning nested datasets/datasets ...")
    for name in os.listdir(nested):
        shutil.move(os.path.join(nested, name), os.path.join(DATASETS_DIR, name))
    os.rmdir(nested)

MIP360_SCENES = [
    "bicycle", "flowers", "garden", "stump", "treehill",
    "room", "counter", "kitchen", "bonsai",
]
IMAGES = "images_4"
MIP360_ROOT = os.path.join(DATASETS_DIR, "mipnerf360")

print(f"Checking {MIP360_ROOT} for {len(MIP360_SCENES)} scenes x {IMAGES} ...")
print()
missing = []
for scene in MIP360_SCENES:
    scene_dir = os.path.join(MIP360_ROOT, scene)
    images_dir = os.path.join(scene_dir, IMAGES)
    if not os.path.isdir(scene_dir):
        status = "MISSING (scene dir not found)"
        missing.append(scene)
    elif not os.path.isdir(images_dir):
        status = f"MISSING ({IMAGES} not found; has: {sorted(os.listdir(scene_dir))[:6]})"
        missing.append(scene)
    else:
        n_imgs = len([f for f in os.listdir(images_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
        status = f"OK ({n_imgs} images)"
    print(f"  {scene:<12s} {status}")

print()
if missing:
    print(f"⚠️  {len(missing)}/{len(MIP360_SCENES)} scene(s) missing {IMAGES}: {missing}")
    print("    run_mip360.sh checks for this dir per-scene and SKIPS (does not fail) if absent.")
    print(f"    Actual layout under {MIP360_ROOT}:")
    if os.path.isdir(MIP360_ROOT):
        for entry in sorted(os.listdir(MIP360_ROOT)):
            print("     -", entry)
    else:
        print("    (mipnerf360 folder not found at all -- check the dataset-copy cell above,")
        print("     and/or attach a Kaggle dataset containing it, then re-run that cell.)")
else:
    print(f"✅ All {len(MIP360_SCENES)} scenes have {IMAGES} present. Safe to run the sweep below.")


In [ ]:
%%bash
# ============================================================
# MIP-NERF 360 FULL SWEEP -- images_4, all 9 scenes (run_mip360.sh)
# ============================================================
# NOTE: this trains all 9 scenes back to back (30000 iters each) -- expect
# several hours total. STOP_ON_ERROR=True is set inside run_mip360.sh, so a
# failure on any one scene halts the remaining scenes; check the tail below
# (or the full log file) to see which scene it stopped on.
export PATH=/opt/conda/bin:$PATH
source /opt/conda/bin/activate
export CUDA_HOME=/opt/conda
export LD_LIBRARY_PATH=/opt/conda/lib:$LD_LIBRARY_PATH
export CUDA_VISIBLE_DEVICES=0

cd /kaggle/working/spec-fastgs/spec-fastgs

# Guard against a stray nested datasets/datasets/ level (see the
# dataset-copy and verification cells above); harmless if already flat.
if [ -d "./datasets/datasets" ]; then
    echo "Re-aligning dataset file structure..."
    mv ./datasets/datasets/* ./datasets/
    rm -rf ./datasets/datasets
fi

LOGFILE=/kaggle/working/mip360_images4_run.log
echo "Running run_mip360.sh (all 9 scenes, images_4) ..."
bash run_mip360.sh > "$LOGFILE" 2>&1
STATUS=$?

echo "--- tail of ${LOGFILE} ---"
tail -n 100 "$LOGFILE"
echo "run_mip360.sh exit status: $STATUS"
if [ $STATUS -ne 0 ]; then
    echo "⚠️  run_mip360.sh stopped early (STOP_ON_ERROR=True) -- see the log above, or the"
    echo "    full log at ${LOGFILE}, for which scene it stopped on."
fi


In [ ]:
# ============================================================
# MIP-NERF 360 SWEEP -- RESULTS SUMMARY
# ============================================================
import json
import os

REPO_ROOT = "/kaggle/working/spec-fastgs/spec-fastgs"
OUTPUT_ROOT = os.path.join(REPO_ROOT, "output", "mip360_images4")

MIP360_SCENES = [
    "bicycle", "flowers", "garden", "stump", "treehill",
    "room", "counter", "kitchen", "bonsai",
]

def fmt(x, nd=4):
    return f"{x:.{nd}f}" if isinstance(x, (int, float)) else "-"

header = f"{'scene':<12s}{'PSNR':>8s}{'SSIM':>8s}{'LPIPS':>8s}{'Spec_PSNR':>11s}{'ASG_IoU':>9s}{'#Gauss':>10s}{'time':>10s}"
print(header)
print("-" * len(header))
for scene in MIP360_SCENES:
    out_dir = os.path.join(OUTPUT_ROOT, scene)
    results_path = os.path.join(out_dir, "results_grouped.json")
    info_path = os.path.join(out_dir, "train_info.json")

    if not os.path.exists(results_path):
        print(f"{scene:<12s}  (no results_grouped.json -- skipped, or sweep did not reach this scene)")
        continue

    with open(results_path) as f:
        results = json.load(f)
    scene_result = next(iter(results.values()))
    render_result = next(iter(scene_result.values()))
    main = render_result.get("main_metrics", {})
    aux = render_result.get("aux_metrics", {})

    info = {}
    if os.path.exists(info_path):
        with open(info_path) as f:
            info = json.load(f)

    print(f"{scene:<12s}{fmt(main.get('PSNR')):>8s}{fmt(main.get('SSIM')):>8s}{fmt(main.get('LPIPS')):>8s}"
          f"{fmt(aux.get('Spec_PSNR')):>11s}{fmt(aux.get('ASG_Residual_IoU')):>9s}"
          f"{str(info.get('final_gaussians', '-')):>10s}{str(info.get('training_time_formatted', '-')):>10s}")


In [ ]:
# ============================================================
# MIP-NERF 360 SWEEP -- ARCHIVE OUTPUTS
# ============================================================
import shutil
import os

REPO_ROOT = "/kaggle/working/spec-fastgs/spec-fastgs"
src = os.path.join(REPO_ROOT, "output", "mip360_images4")
out = "/kaggle/working/spec_fastgs_output_mip360_images4"

if os.path.isdir(src):
    shutil.make_archive(out, "zip", src)
    print("archived:", out + ".zip", round(os.path.getsize(out + ".zip") / 1e6, 1), "MB")
else:
    print("no mip360_images4 output found at", src)
